In [1]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm import tqdm
import numpy as np

import timm

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

In [2]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

dataset_path = "/kaggle/input/ai-medleafx"
resized_path = os.path.join(dataset_path, "Resized Image")

print("Using dataset:", resized_path)

# Standard ViT MoCo v3 transforms
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )
])

Using dataset: /kaggle/input/ai-medleafx/Resized Image


In [3]:
class LeafDataset(Dataset):
    def __init__(self, root_dir, transform):
        self.samples = []
        self.transform = transform

        plant_types = sorted(os.listdir(root_dir))
        
        for plant in plant_types:
            plant_path = os.path.join(root_dir, plant)
            if not os.path.isdir(plant_path):
                continue
            
            statuses = sorted(os.listdir(plant_path))
            for status in statuses:
                class_name = f"{plant}_{status}"
                class_path = os.path.join(plant_path, status)

                for img in os.listdir(class_path):
                    if img.lower().endswith((".jpg", ".jpeg", ".png", ".JPG")):
                        self.samples.append(
                            (os.path.join(class_path, img), class_name)
                        )

        # Encode labels
        class_names = sorted(list(set([c for _, c in self.samples])))
        self.class_to_idx = {cls: i for i, cls in enumerate(class_names)}

        self.samples = [
            (path, self.class_to_idx[label]) for path, label in self.samples
        ]

        print("Total images:", len(self.samples))
        print("Classes:", len(self.class_to_idx))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = Image.open(img_path).convert("RGB")
        img = self.transform(img)
        return img, label

In [4]:
full_dataset = LeafDataset(resized_path, transform)

# Split manually (e.g., 80% train, 20% test)
train_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_size

train_ds, test_ds = torch.utils.data.random_split(
    full_dataset, [train_size, test_size]
)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_ds, batch_size=32, shuffle=False)

Total images: 10858
Classes: 13


In [5]:
model = timm.create_model(
    "hf_hub:1aurent/vit_small_patch16_224.transpath_mocov3",
    pretrained=True,
    num_classes=0
).to(DEVICE)

print("Model loaded. Feature dim =", model.num_features)

config.json:   0%|          | 0.00/568 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/86.7M [00:00<?, ?B/s]

Model loaded. Feature dim = 384


In [6]:
def extract_features(model_instance, dataloader, device):
    """
    Extract CLS token features from MoCo-v3 ViT.

    Returns:
        all_features: np.array of shape [num_samples, feature_dim]
        all_labels: np.array of shape [num_samples]
    """
    model_instance.eval()
    all_features = []
    all_labels = []

    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc="Extracting features"):
            images = images.to(device)

            feats = model_instance.forward_features(images)  # [batch, seq_len, feat_dim] or [batch, feat_dim]

            # FIX: Take CLS token if 3D
            if feats.ndim == 3:
                feats = feats[:, 0, :]  # CLS token → shape [batch, feature_dim]

            all_features.append(feats.cpu().numpy())
            all_labels.append(labels.numpy())

    all_features = np.concatenate(all_features, axis=0)  # shape [num_samples, feature_dim]
    all_labels = np.concatenate(all_labels, axis=0)
    return all_features, all_labels

In [7]:
train_X, train_y = extract_features(model, train_loader, DEVICE)
test_X, test_y   = extract_features(model, test_loader, DEVICE)

print("Train feature shape:", train_X.shape)  # Should be [num_samples, feature_dim]
print("Test feature shape:", test_X.shape)    # Should be [num_samples, feature_dim]

Extracting features: 100%|██████████| 68/68 [00:25<00:00,  2.65it/s]

Train feature shape: (8686, 384)
Test feature shape: (2172, 384)


In [8]:
scaler = StandardScaler()
train_X = scaler.fit_transform(train_X)  # Now works correctly
test_X  = scaler.transform(test_X)

In [9]:
pca = PCA(n_components=384)
train_X = pca.fit_transform(train_X)
test_X  = pca.transform(test_X)

In [10]:
mlp = MLPClassifier(
    hidden_layer_sizes=(1024, 512, 256),
    activation='relu',
    solver='adam',
    learning_rate_init=1e-4,
    max_iter=800,
    random_state=42,
    verbose=True
)

mlp.fit(train_X, train_y)

Iteration 1, loss = 1.51294007
Iteration 2, loss = 0.49489846
Iteration 3, loss = 0.29263137
Iteration 4, loss = 0.21462429
Iteration 5, loss = 0.16751197
Iteration 6, loss = 0.13571380
Iteration 7, loss = 0.11265662
Iteration 8, loss = 0.09316680
Iteration 9, loss = 0.07668866
Iteration 10, loss = 0.06555632
Iteration 11, loss = 0.05502350
Iteration 12, loss = 0.04604145
Iteration 13, loss = 0.04003298
Iteration 14, loss = 0.03388323
Iteration 15, loss = 0.02971291
Iteration 16, loss = 0.02614810
Iteration 17, loss = 0.02257922
Iteration 18, loss = 0.02103067
Iteration 19, loss = 0.01836645
Iteration 20, loss = 0.01775063
Iteration 21, loss = 0.01579825
Iteration 22, loss = 0.01591323
Iteration 23, loss = 0.01433070
Iteration 24, loss = 0.01352172
Iteration 25, loss = 0.01310786
Iteration 26, loss = 0.01204838
Iteration 27, loss = 0.01131610
Iteration 28, loss = 0.01145437
Iteration 29, loss = 0.01030974
Iteration 30, loss = 0.01318842
Iteration 31, loss = 0.01037929
Iteration 32, los

MLPClassifier(hidden_layer_sizes=(1024, 512, 256), learning_rate_init=0.0001,
              max_iter=800, random_state=42, verbose=True)

In [11]:
preds = mlp.predict(test_X)
acc = accuracy_score(test_y, preds)

print(f"\n✅ MLP Test Accuracy: {acc * 100:.2f}%")


✅ MLP Test Accuracy: 94.43%
